# 15 - Train external semantic adapters with a frozen shared codebook

The Symile bridge and shared codebook remain frozen. A small residual adapter is learned for each external sensor, while either one shared or two separate text projectors map diagnoses/reports into the same space. Training uses external train pairs; validation selects checkpoints.

Raw shared-text adapters are trained for every bridge variant. The two deeper semantic ablations (`separate_raw_text` and `shared_concepts`) are run for `shared_specific_vq`. Sensor and text status masks are intersected, so partial or failed encodings never enter training. Repeated identical texts/concept lists are treated as multiple positives rather than false negatives.

Finished runs are validated and skipped automatically. A completion marker records the exact bridge, data signatures, semantic configuration, and training hyperparameters; interrupted or stale runs are retrained. Set `SEMANTIC_FORCE_RETRAIN=1` to deliberately rerun completed configurations. A small smoke run can be selected with `SEMANTIC_BRIDGE_VARIANTS`, `SEMANTIC_VARIANTS`, `SEMANTIC_ADAPTER_SEEDS`, and `SEMANTIC_ADAPTER_EPOCHS`.


In [ ]:
from pathlib import Path
import hashlib
import importlib
import json
import os
import random
import sys

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

ARTIFACT_ROOT = Path(os.environ.get("ECG_CXR_BRIDGE_ARTIFACT_ROOT", "/data/liangz2/ecg_analysis/symile/ecg_cxr_bridge_artifacts"))
CODE_ROOT = ARTIFACT_ROOT / "code"
module_path = CODE_ROOT / "ecg_cxr_ablation.py"
if not module_path.is_file():
    raise FileNotFoundError(f"Run notebook 12 first; missing {module_path}")
if str(CODE_ROOT) not in sys.path: sys.path.insert(0, str(CODE_ROOT))
import ecg_cxr_ablation
importlib.reload(ecg_cxr_ablation)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_rows(modality):
    path = ARTIFACT_ROOT / "external" / modality / "manifest.jsonl"
    if not path.is_file():
        raise FileNotFoundError(f"Run notebook {'10' if modality == 'ecg' else '11'} first; missing {path}")
    values = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    if not values:
        raise ValueError(f"Empty {modality} manifest: {path}")
    required = ["sample_id", "split", "machine_diagnosis" if modality == "ecg" else "report_text"]
    invalid = [(index, [key for key in required if key not in row or row[key] in (None, "")])
               for index, row in enumerate(values)]
    invalid = [(index, missing) for index, missing in invalid if missing]
    if invalid:
        raise KeyError(f"{modality} manifest rows missing required fields: {invalid[:20]}")
    return values

def sensor_manifest_sha256(values):
    payload = json.dumps(values, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return hashlib.sha256(payload.encode()).hexdigest()

def text_manifest_sha256(values):
    digest = hashlib.sha256()
    for row in values:
        encoded = json.dumps(row, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode()
        digest.update(len(encoded).to_bytes(8, "big")); digest.update(encoded)
    return digest.hexdigest()

def load_concept_texts(modality):
    path = ARTIFACT_ROOT / "external" / modality / "concepts.jsonl"
    if not path.is_file():
        raise FileNotFoundError(f"Run notebook 14 first; missing {path}")
    values = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    if len(values) != len(rows[modality]):
        raise ValueError(f"{modality} concept rows ({len(values)}) do not match manifest ({len(rows[modality])})")
    for index, (row, concept) in enumerate(zip(rows[modality], values)):
        if str(row["sample_id"]) != str(concept.get("sample_id")):
            raise ValueError(f"{modality} concept order mismatch at row {index}")
    return [str(value.get("concept_text", "")).strip() for value in values]

rows = {m: load_rows(m) for m in ["ecg", "cxr"]}
manifest_hashes = {
    m: {"sensor": sensor_manifest_sha256(rows[m]), "text": text_manifest_sha256(rows[m])}
    for m in ["ecg", "cxr"]}
concept_texts = {m: load_concept_texts(m) for m in ["ecg", "cxr"]}
experiment_path = ARTIFACT_ROOT / "ablation" / "experiment_index.json"
if not experiment_path.is_file():
    raise FileNotFoundError(f"Run notebook 12 first; missing {experiment_path}")
experiment_index = json.loads(experiment_path.read_text())
if not isinstance(experiment_index, list) or not experiment_index:
    raise ValueError(f"No bridge experiments in {experiment_path}")

selected_bridge_variants = {value.strip() for value in os.environ.get(
    "SEMANTIC_BRIDGE_VARIANTS", "").split(",") if value.strip()}
if selected_bridge_variants:
    available = {item["variant"] for item in experiment_index}
    unknown = selected_bridge_variants - available
    if unknown:
        raise ValueError(f"Unknown SEMANTIC_BRIDGE_VARIANTS {sorted(unknown)}; available={sorted(available)}")
    experiment_index = [item for item in experiment_index if item["variant"] in selected_bridge_variants]
selected_bridge_seeds = {int(value.strip()) for value in os.environ.get(
    "SEMANTIC_BRIDGE_SEEDS", "").split(",") if value.strip()}
if selected_bridge_seeds:
    available_seeds = {int(item["seed"]) for item in experiment_index}
    unknown_seeds = selected_bridge_seeds - available_seeds
    if unknown_seeds:
        raise ValueError(f"Unknown SEMANTIC_BRIDGE_SEEDS {sorted(unknown_seeds)}; available={sorted(available_seeds)}")
    experiment_index = [item for item in experiment_index if int(item["seed"]) in selected_bridge_seeds]
if not experiment_index:
    raise ValueError("No bridge experiments selected")

# Verify the source manifests are patient-disjoint across training and validation.
for modality in ["ecg", "cxr"]:
    subject_sets = {split: {int(row["subject_id"]) for row in rows[modality]
                            if row["split"] == split and row.get("subject_id") is not None}
                    for split in ["train", "val"]}
    overlap = subject_sets["train"] & subject_sets["val"]
    if overlap:
        raise AssertionError(f"Patient leakage in external {modality} train/val: {sorted(overlap)[:20]}")
print({"device": str(DEVICE), "bridge_runs": len(experiment_index),
       "manifest_rows": {m: len(rows[m]) for m in rows}})


In [ ]:
def load_json(path):
    try:
        value = json.loads(path.read_text())
        return value if isinstance(value, dict) else {}
    except (FileNotFoundError, json.JSONDecodeError):
        return {}

class ExternalPairs(Dataset):
    def __init__(self, modality, experiment, split, text_view):
        self.modality, self.rows = modality, rows[modality]
        base = ARTIFACT_ROOT / "external" / modality
        bridge = base / "bridge" / experiment["variant"] / f"seed_{experiment['seed']}"
        paths = {
            "sensor": bridge / "continuous_common.npy",
            "text": base / f"text_{text_view}_biomedclip.npy",
            "sensor_status": base / "encoding_status.npy",
            "text_status": base / f"text_{text_view}_encoding_status.npy",
            "bridge_metadata": bridge / "metadata.json",
            "text_metadata": base / f"text_{text_view}_metadata.json",
        }
        missing = [str(path) for path in paths.values() if not path.is_file()]
        if missing:
            raise FileNotFoundError("Run notebooks 13-14 first; missing:\n" + "\n".join(missing))
        bridge_metadata = load_json(paths["bridge_metadata"])
        text_metadata = load_json(paths["text_metadata"])
        if bridge_metadata.get("manifest_sha256") != manifest_hashes[modality]["sensor"]:
            raise ValueError(f"Stale {modality} bridge features for {experiment['variant']} seed {experiment['seed']}; rerun notebook 13")
        if text_metadata.get("manifest_sha256") != manifest_hashes[modality]["text"]:
            raise ValueError(f"Stale {modality} {text_view} text features; rerun notebook 14")
        self.sensor = np.load(paths["sensor"], mmap_mode="r")
        self.text = np.load(paths["text"], mmap_mode="r")
        sensor_status = np.load(paths["sensor_status"], mmap_mode="r")
        text_status = np.load(paths["text_status"], mmap_mode="r")
        expected_rows = len(self.rows)
        lengths = {"sensor": len(self.sensor), "text": len(self.text),
                   "sensor_status": len(sensor_status), "text_status": len(text_status)}
        if any(value != expected_rows for value in lengths.values()):
            raise ValueError(f"{modality} array/manifest row mismatch: manifest={expected_rows}, arrays={lengths}")
        if self.sensor.ndim != 2 or self.text.ndim != 2:
            raise ValueError(f"Expected 2-D {modality} arrays, got sensor={self.sensor.shape}, text={self.text.shape}")
        self.index = np.asarray([
            i for i, row in enumerate(self.rows)
            if row["split"] == split and sensor_status[i] == 1 and text_status[i] == 1
        ], dtype="int64")
        raw_key = "machine_diagnosis" if modality == "ecg" else "report_text"
        group_text = concept_texts[modality] if text_view == "concept" else [
            " ".join(str(row[raw_key]).lower().split()) for row in self.rows]
        self.group = np.asarray([
            int(hashlib.sha1(group_text[i].encode()).hexdigest()[:15], 16) for i in self.index
        ], dtype="int64")
        self.subject = np.asarray([
            int(self.rows[i]["subject_id"]) if self.rows[i].get("subject_id") is not None else -(int(i) + 1)
            for i in self.index], dtype="int64")
        for start in range(0, len(self.index), 4096):
            take = self.index[start:start + 4096]
            if not np.isfinite(np.asarray(self.sensor[take])).all() or not np.isfinite(np.asarray(self.text[take])).all():
                raise ValueError(f"Non-finite {modality} {split}/{text_view} features at selected rows")
    def __len__(self): return len(self.index)
    def __getitem__(self, position):
        index = self.index[position]
        return {"sensor": torch.tensor(np.array(self.sensor[index], dtype="float32")),
                "text": torch.tensor(np.array(self.text[index], dtype="float32")),
                "group": torch.tensor(self.group[position]), "subject": torch.tensor(self.subject[position]),
                "row_index": torch.tensor(index)}

class Adapter(nn.Module):
    def __init__(self, bridge, text_dim, shared_text=True):
        super().__init__(); self.bridge = bridge; self.shared_text = shared_text
        for parameter in self.bridge.parameters(): parameter.requires_grad = False
        dim = bridge.common_dim
        self.residual = nn.ModuleDict({m: nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim)) for m in ["ecg", "cxr"]})
        for module in self.residual.values(): nn.init.zeros_(module[-1].weight); nn.init.zeros_(module[-1].bias)
        names = ["shared"] if shared_text else ["ecg", "cxr"]
        self.text_projector = nn.ModuleDict({name: nn.Sequential(nn.LayerNorm(text_dim), nn.Linear(text_dim, dim)) for name in names})
    def quantize(self, values):
        if not self.bridge.uses_vq: return F.normalize(values, dim=-1)
        quantized, _, _, _ = self.bridge.common_codebook(values)
        return F.normalize(quantized, dim=-1)
    def sensor(self, modality, values): return self.quantize(values + self.residual[modality](values))
    def text(self, modality, values):
        key = "shared" if self.shared_text else modality
        return self.quantize(self.text_projector[key](values))

    def train(self, mode=True):
        super().train(mode)
        self.bridge.eval()
        return self

def multi_positive_direction(logits, positives, ignored):
    logits = logits.masked_fill(ignored & ~positives, -torch.inf)
    positive_logits = logits.masked_fill(~positives, -torch.inf)
    if not positives.any(1).all():
        raise RuntimeError("Every query must have at least one positive")
    return -(torch.logsumexp(positive_logits, dim=1) - torch.logsumexp(logits, dim=1)).mean()

def contrastive(left, right, group, subject, temperature=.07):
    logits = left @ right.T / temperature
    same_text = group[:, None].eq(group[None, :])
    same_subject = subject[:, None].eq(subject[None, :]) & subject[:, None].ge(0)
    return .5 * (multi_positive_direction(logits, same_text, same_subject)
                 + multi_positive_direction(logits.T, same_text.T, same_subject.T))


In [ ]:
semantic_variants = []
for experiment in experiment_index:
    semantic_variants.append((experiment, "shared_raw_text", "raw", True))
    if experiment["variant"] == "shared_specific_vq":
        semantic_variants.extend([(experiment, "separate_raw_text", "raw", False),
                                  (experiment, "shared_concepts", "concept", True)])
selected_semantic = {value.strip() for value in os.environ.get(
    "SEMANTIC_VARIANTS", "").split(",") if value.strip()}
available_semantic = {value[1] for value in semantic_variants}
if selected_semantic:
    unknown = selected_semantic - available_semantic
    if unknown:
        raise ValueError(f"Unknown SEMANTIC_VARIANTS {sorted(unknown)}; available={sorted(available_semantic)}")
    semantic_variants = [value for value in semantic_variants if value[1] in selected_semantic]
if not semantic_variants:
    raise ValueError("No semantic adapter experiments selected")

seeds = [int(v.strip()) for v in os.environ.get("SEMANTIC_ADAPTER_SEEDS", "43,44,45").split(",") if v.strip()]
EPOCHS = int(os.environ.get("SEMANTIC_ADAPTER_EPOCHS", "30"))
BATCH = int(os.environ.get("SEMANTIC_ADAPTER_BATCH_SIZE", "256"))
PATIENCE = int(os.environ.get("SEMANTIC_ADAPTER_PATIENCE", "5"))
NUM_WORKERS = int(os.environ.get("SEMANTIC_ADAPTER_NUM_WORKERS", "0"))
LEARNING_RATE = float(os.environ.get("SEMANTIC_ADAPTER_LR", "3e-4"))
WEIGHT_DECAY = float(os.environ.get("SEMANTIC_ADAPTER_WEIGHT_DECAY", "1e-4"))
GRAD_CLIP = float(os.environ.get("SEMANTIC_ADAPTER_GRAD_CLIP", "1.0"))
TEMPERATURE = float(os.environ.get("SEMANTIC_ADAPTER_TEMPERATURE", "0.07"))
FORCE_RETRAIN = os.environ.get("SEMANTIC_FORCE_RETRAIN", "0").strip().lower() in {"1", "true", "yes"}
if not seeds or EPOCHS < 1 or BATCH < 2 or PATIENCE < 1 or NUM_WORKERS < 0:
    raise ValueError("Seeds must be nonempty; epochs/patience >= 1; batch >= 2; workers >= 0")
if LEARNING_RATE <= 0 or WEIGHT_DECAY < 0 or GRAD_CLIP <= 0 or TEMPERATURE <= 0:
    raise ValueError("Invalid optimizer, gradient clipping, or temperature setting")

def load_checkpoint(path, map_location=DEVICE):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

training_config = {
    "epochs": EPOCHS, "batch_size": BATCH, "patience": PATIENCE,
    "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
    "gradient_clip": GRAD_CLIP, "temperature": TEMPERATURE,
}

def run_key(item):
    return (str(item["bridge_variant"]), int(item["bridge_seed"]),
            str(item["semantic_variant"]), int(item["adapter_seed"]))

def run_signature(experiment, semantic_name, text_view, shared_text, seed):
    return {
        "bridge_variant": str(experiment["variant"]),
        "bridge_seed": int(experiment["seed"]),
        "bridge_checkpoint": str(Path(experiment["checkpoint"]).resolve()),
        "semantic_variant": str(semantic_name), "text_view": str(text_view),
        "shared_text": bool(shared_text), "adapter_seed": int(seed),
        "training_config": training_config,
        "manifest_hashes": manifest_hashes,
    }

def completed_run(run_dir, signature):
    if FORCE_RETRAIN:
        return None
    best_path = run_dir / "best.pt"
    history_path = run_dir / "history.json"
    completion_path = run_dir / "completed.json"
    if not best_path.is_file() or not history_path.is_file():
        return None
    marker = load_json(completion_path)
    if marker:
        if marker.get("status") != "complete" or marker.get("signature") != signature:
            return None
    try:
        history = json.loads(history_path.read_text())
        if not isinstance(history, list) or not history:
            return None
        validation = [float(row["val"]) for row in history]
        if not np.isfinite(validation).all():
            return None
        state = load_checkpoint(best_path, map_location="cpu")
        experiment = state.get("experiment", {})
        state_config = state.get("training_config", {})
        matches = (
            str(experiment.get("variant")) == signature["bridge_variant"]
            and int(experiment.get("seed", -1)) == signature["bridge_seed"]
            and str(Path(state.get("bridge_checkpoint", "")).resolve()) == signature["bridge_checkpoint"]
            and state.get("semantic_variant") == signature["semantic_variant"]
            and state.get("text_view") == signature["text_view"]
            and bool(state.get("shared_text")) == signature["shared_text"]
            and int(state.get("seed", -1)) == signature["adapter_seed"]
            and all(state_config.get(key) == value for key, value in training_config.items())
            and isinstance(state.get("model"), dict) and bool(state["model"])
        )
        if not matches:
            return None
        best_validation = float(min(validation))
        entry = {
            "bridge_variant": signature["bridge_variant"],
            "bridge_seed": signature["bridge_seed"],
            "semantic_variant": signature["semantic_variant"],
            "adapter_seed": signature["adapter_seed"],
            "best_validation_loss": best_validation,
            "checkpoint": str(best_path), "text_view": signature["text_view"],
            "data_counts": state.get("data_counts", {}),
        }
        # Legacy notebook-15 runs wrote history only after the loop completed.
        # Once validated, give them the same explicit completion marker.
        if not marker:
            completion_path.write_text(json.dumps({
                "status": "complete", "signature": signature,
                "epochs_completed": len(history),
                "best_validation_loss": best_validation,
                "recovered_legacy_run": True,
            }, indent=2))
        return entry
    except (OSError, ValueError, TypeError, KeyError, json.JSONDecodeError) as exc:
        print("Existing adapter is incomplete or invalid; retraining:", run_dir, repr(exc))
        return None

index_path = ARTIFACT_ROOT / "external" / "adapters" / "adapter_index.json"
try:
    existing_adapter_index = json.loads(index_path.read_text())
    if not isinstance(existing_adapter_index, list):
        existing_adapter_index = []
except (FileNotFoundError, json.JSONDecodeError):
    existing_adapter_index = []
adapter_index = []
selected_run_keys = set()
for experiment, semantic_name, text_view, shared_text in semantic_variants:
    checkpoint_path = Path(experiment["checkpoint"])
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Missing bridge checkpoint: {checkpoint_path}")
    checkpoint = load_checkpoint(checkpoint_path)
    resolved = checkpoint["resolved_config"]
    for seed in seeds:
        run_dir = (ARTIFACT_ROOT / "external" / "adapters" / experiment["variant"]
                   / f"bridge_seed_{experiment['seed']}" / semantic_name / f"seed_{seed}")
        run_dir.mkdir(parents=True, exist_ok=True)
        signature = run_signature(experiment, semantic_name, text_view, shared_text, seed)
        selected_run_keys.add(run_key(signature))
        finished_entry = completed_run(run_dir, signature)
        if finished_entry is not None:
            adapter_index.append(finished_entry)
            print("SKIP completed adapter:", experiment["variant"], experiment["seed"],
                  semantic_name, seed, "best_val=", finished_entry["best_validation_loss"])
            continue

        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
        bridge = ecg_cxr_ablation.AblationBridge(resolved["input_dims"], resolved["model_variant"], **resolved["model_hparams"]).to(DEVICE)
        bridge.load_state_dict(checkpoint["model"]); bridge.eval()
        datasets = {split: {m: ExternalPairs(m, experiment, split, text_view) for m in ["ecg", "cxr"]} for split in ["train", "val"]}
        counts = {split: {m: len(datasets[split][m]) for m in ["ecg", "cxr"]} for split in ["train", "val"]}
        too_small = [(split, modality, count) for split, values in counts.items()
                     for modality, count in values.items() if count < 2]
        if too_small:
            raise ValueError(f"Need at least two jointly encoded rows per train/val modality; found {too_small}. Complete notebooks 13-14.")
        insufficient_groups = [(split, modality, int(np.unique(datasets[split][modality].group).size))
                               for split in ["train", "val"] for modality in ["ecg", "cxr"]
                               if np.unique(datasets[split][modality].group).size < 2]
        if insufficient_groups:
            raise ValueError(f"Contrastive training/validation requires at least two distinct texts: {insufficient_groups}")
        text_dims = {m: int(datasets["train"][m].text.shape[1]) for m in ["ecg", "cxr"]}
        if len(set(text_dims.values())) != 1:
            raise ValueError(f"ECG/CXR text dimensions differ: {text_dims}")
        text_dim = text_dims["ecg"]
        expected_common_dim = int(resolved["model_hparams"]["common_dim"])
        sensor_dims = {m: int(datasets["train"][m].sensor.shape[1]) for m in ["ecg", "cxr"]}
        if set(sensor_dims.values()) != {expected_common_dim}:
            raise ValueError(f"Continuous shared-feature dimensions do not match bridge common_dim: {sensor_dims}")
        model = Adapter(bridge, text_dim, shared_text).to(DEVICE)
        optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                      lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

        loader_generator = torch.Generator().manual_seed(seed)
        train_loaders = {m: DataLoader(
            datasets["train"][m], batch_size=BATCH, shuffle=True,
            drop_last=len(datasets["train"][m]) >= BATCH,
            num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
            persistent_workers=NUM_WORKERS > 0, generator=loader_generator,
        ) for m in ["ecg", "cxr"]}
        val_loaders = {m: DataLoader(
            datasets["val"][m], batch_size=BATCH, shuffle=False, drop_last=False,
            num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
            persistent_workers=NUM_WORKERS > 0,
        ) for m in ["ecg", "cxr"]}

        def batch_loss(modality, batch):
            sensor, text, group, subject = [batch[k].to(DEVICE, non_blocking=True)
                                            for k in ["sensor", "text", "group", "subject"]]
            return contrastive(model.sensor(modality, sensor), model.text(modality, text),
                               group, subject, TEMPERATURE)

        def train_epoch():
            model.train(True); losses = []
            iterators = {m: iter(train_loaders[m]) for m in ["ecg", "cxr"]}
            for _ in range(max(len(train_loaders["ecg"]), len(train_loaders["cxr"]))):
                terms = []
                for modality in ["ecg", "cxr"]:
                    try:
                        batch = next(iterators[modality])
                    except StopIteration:
                        iterators[modality] = iter(train_loaders[modality])
                        batch = next(iterators[modality])
                    if len(batch["sensor"]) >= 2:
                        terms.append(batch_loss(modality, batch))
                if not terms:
                    continue
                optimizer.zero_grad(set_to_none=True)
                loss = torch.stack(terms).mean(); loss.backward()
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
                optimizer.step(); losses.append(float(loss.detach().cpu()))
            if not losses:
                raise RuntimeError("No usable training batches; reduce SEMANTIC_ADAPTER_BATCH_SIZE")
            return float(np.mean(losses))

        @torch.inference_mode()
        def validation_epoch():
            model.eval(); modality_losses = []
            for modality in ["ecg", "cxr"]:
                weighted, total_rows = 0.0, 0
                for batch in val_loaders[modality]:
                    if len(batch["sensor"]) < 2:
                        continue
                    loss = float(batch_loss(modality, batch).cpu())
                    weighted += loss * len(batch["sensor"]); total_rows += len(batch["sensor"])
                if total_rows == 0:
                    raise RuntimeError(f"No usable {modality} validation batches")
                modality_losses.append(weighted / total_rows)
            return float(np.mean(modality_losses))

        completion_path = run_dir / "completed.json"
        completion_path.write_text(json.dumps({
            "status": "running", "signature": signature,
            "message": "A running marker is never treated as a completed adapter."
        }, indent=2))
        best, stale, history = float("inf"), 0, []
        for ep in range(EPOCHS):
            tr, va = train_epoch(), validation_epoch()
            if not np.isfinite(tr) or not np.isfinite(va):
                raise FloatingPointError(f"Non-finite adapter loss: train={tr}, val={va}")
            history.append({"epoch": ep, "train": tr, "val": va})
            print(experiment["variant"], experiment["seed"], semantic_name, seed, history[-1])
            state = {"model": model.state_dict(), "bridge_checkpoint": experiment["checkpoint"],
                     "experiment": experiment, "semantic_variant": semantic_name, "text_view": text_view,
                     "shared_text": shared_text, "text_dim": text_dim, "seed": seed, "epoch": ep,
                     "data_counts": counts,
                     "training_config": training_config}
            if va < best: best, stale = va, 0; torch.save(state, run_dir / "best.pt")
            else:
                stale += 1
                if stale >= PATIENCE: break
        (run_dir / "history.json").write_text(json.dumps(history, indent=2))
        if not (run_dir / "best.pt").is_file():
            raise RuntimeError(f"No adapter checkpoint was saved in {run_dir}")
        completion_reason = "early_stopping" if stale >= PATIENCE else "maximum_epochs"
        completion_path.write_text(json.dumps({
            "status": "complete", "signature": signature,
            "epochs_completed": len(history), "completion_reason": completion_reason,
            "best_validation_loss": best,
        }, indent=2))
        adapter_index.append({
            "bridge_variant": experiment["variant"], "bridge_seed": experiment["seed"],
            "semantic_variant": semantic_name, "adapter_seed": seed,
            "best_validation_loss": best, "checkpoint": str(run_dir / "best.pt"),
            "text_view": text_view, "data_counts": counts})
        del model, bridge
        if torch.cuda.is_available(): torch.cuda.empty_cache()
preserved = []
for item in existing_adapter_index:
    try:
        if run_key(item) not in selected_run_keys and Path(item["checkpoint"]).is_file():
            preserved.append(item)
    except (KeyError, TypeError, ValueError):
        continue
merged = {}
for item in preserved + adapter_index:
    merged[run_key(item)] = item
final_adapter_index = [merged[key] for key in sorted(merged)]
index_path.parent.mkdir(parents=True, exist_ok=True)
index_path.write_text(json.dumps(final_adapter_index, indent=2))
print({"selected_runs": len(selected_run_keys), "completed_or_trained": len(adapter_index),
       "preserved_unselected": len(preserved), "index_entries": len(final_adapter_index)})
print("Saved:", index_path)
